In [1]:
import pandas as pd
import plotly.express as px
from movie_analytics.database import get_engine

In [2]:
engine = get_engine()
movies_df = pd.read_sql("SELECT * FROM movies_full_dataset",engine)
movies_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6431 entries, 0 to 6430
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   imdb_id     6431 non-null   str    
 1   title       6431 non-null   str    
 2   year        6431 non-null   int64  
 3   genres      6429 non-null   str    
 4   avg_rating  6431 non-null   float64
 5   votes       6431 non-null   int64  
 6   popularity  6431 non-null   float64
 7   revenue     6431 non-null   int64  
 8   budget      6431 non-null   int64  
dtypes: float64(2), int64(4), str(3)
memory usage: 452.3 KB


In [3]:
df_cleaned = movies_df.dropna(subset=["genres"])
df_cleaned["genres_list"] = df_cleaned["genres"].str.split(',')
df_cleaned["genres_weight"] = 1/df_cleaned["genres_list"].apply(len)
df_explode = df_cleaned.explode("genres_list")
df = df_explode.rename(columns={"genres_list":"genre","genres_weight":"genre_weight"})
df


,imdb_id,title,year,genres,avg_rating,votes,popularity,revenue,budget,genre,genre_weight
0,tt0468569,The Dark Knight,2008,"Action,Crime,Drama",9.1,3162226,130.6430,1004558444,185000000,Action,0.333333
0,tt0468569,The Dark Knight,2008,"Action,Crime,Drama",9.1,3162226,130.6430,1004558444,185000000,Crime,0.333333
0,tt0468569,The Dark Knight,2008,"Action,Crime,Drama",9.1,3162226,130.6430,1004558444,185000000,Drama,0.333333
1,tt1375666,Inception,2010,"Action,Adventure,Sci-Fi",8.8,2811917,83.9520,825532764,160000000,Action,0.333333
1,tt1375666,Inception,2010,"Action,Adventure,Sci-Fi",8.8,2811917,83.9520,825532764,160000000,Adventure,0.333333
...,...,...,...,...,...,...,...,...,...,...,...
6426,tt36955396,Mamá quiero ser futbolista profesional,2022,Documentary,9.7,7,0.0357,3000,15000,Documentary,1.000000
6427,tt9536384,Mai Mai Ti's 2008,2008,Drama,6.8,7,0.6000,20707,414147,Drama,1.000000
6428,tt11697356,Corpus Peccati,2019,Horror,6.8,7,0.0000,35,3,Horror,1.000000
6429,tt5492684,Tanz der Zuckerfee,2016,Drama,8.7,7,0.6000,2000,10000,Drama,1.000000


In [4]:
genres_list = ["Action", "Adventure", "Animation", "Comedy", "Drama", "Fantasy", "Horror", "Mystery", "Romance", "Thriller", "Sci-Fi"]
ratings_df = df
ratings_df["weighted_value"] = ratings_df["avg_rating"]*ratings_df["genre_weight"]
ratings_df["weight"] = ratings_df["genre_weight"]
genre_ratings_df = df.groupby("genre", as_index=False).agg(sum_weighted_value=("weighted_value","sum"),sum_weight=("weight","sum"),sum_votes=("votes","sum"))
genre_ratings_df = genre_ratings_df.assign(weighted_rating=genre_ratings_df["sum_weighted_value"]/genre_ratings_df["sum_weight"])[["genre","weighted_rating","sum_votes"]]
genre_ratings_df = genre_ratings_df[genre_ratings_df["genre"].isin(genres_list)]
genre_ratings_df.to_csv("genre_ratings_df.csv", index=False)

colors = [
    "#636EFA", # Royal Blue
    "#EF553B", # Sunset Red
    "#00CC96", # Emerald Green
    "#AB63FA", # Vivid Purple
    "#FFA15A", # Safety Orange
    "#19D3F3", # Sky Blue
    "#FF6692", # Hot Pink
    "#B6E880", # Chartreuse
    "#FFD700", # Gold
    "#72B7B2", # Sage/Grey-Teal
    "#542788"  # Deep Indigo
]

fig=px.bar(genre_ratings_df,
        x="genre", 
        y="weighted_rating",
        color="genre",
        color_discrete_sequence=colors
    )
fig.update_yaxes(range=[5,7])

fig.show()




In [5]:
genres_list = ["Action", "Adventure", "Animation", "Comedy", "Drama", "Fantasy", "Horror", "Mystery", "Romance", "Thriller", "Sci-Fi"]

df_sum = df.groupby(["genre","year"],as_index=False).agg(sum_revenue=("revenue","sum"))[["year","genre","sum_revenue"]]
df_sum = df_sum[df_sum["genre"].isin(genres_list)]

colors = [
    "#636EFA", # Royal Blue
    "#EF553B", # Sunset Red
    "#00CC96", # Emerald Green
    "#AB63FA", # Vivid Purple
    "#FFA15A", # Safety Orange
    "#19D3F3", # Sky Blue
    "#FF6692", # Hot Pink
    "#B6E880", # Chartreuse
    "#FFD700", # Gold
    "#72B7B2", # Sage/Grey-Teal
    "#542788"  # Deep Indigo
]

fig = px.line(
    df_sum,
    x="year",
    y="sum_revenue",
    color="genre",
    color_discrete_sequence=colors
)

fig.update_layout(
    legend_itemclick="toggleothers", 
    legend_itemdoubleclick="toggle"      
)

fig.show()